# 01 · MongoDB – Data Modeling, Loading & Aggregation Pipelines
**Part 1.1 of the assignment (25 % shared with HBase).**

**Prerequisite:** MongoDB is running (`docker compose up -d mongo` – see README) and notebook 00 has generated the data.

## Why MongoDB for this data?
MongoDB stores *documents* – nested JSON. Three entities in this dataset are naturally nested:

| Entity | Natural shape | Modelling choice |
|---|---|---|
| **Transaction** | one order → many line items | **Embed** `items[]` (items are never read without their order) |
| **Session** | one visit → ordered page views | **Embed** `page_views[]` (a journey is always read as a whole) |
| **Product** | belongs to a category | **Embed a small copy** of the category (name, margin) – products are read far more often than categories change |
| **User** | profile + behaviour summary | **Embed** `purchase_summary` + `preferences` (pre-computed, "frequently accessed information") |

### The 5 modelling rules we apply (and will justify in the report)
1. **Model for the queries**, not for the data shape: all our analytics are read-heavy aggregations.
2. **Embed** data that is read together and has bounded size; **reference** data that is shared or unbounded.
3. **Denormalise** small, rarely-changing attributes (category name / margin / category_id on each item) to avoid `$lookup` in hot paths.
4. **Replace dynamic keys by arrays** – the raw `cart_contents: {"prod_00123": {...}}` uses product ids as *field names*; MongoDB cannot index or aggregate such keys efficiently (*attribute pattern*) → we convert to `cart_items: [{product_id, quantity, price}]`.
5. **Store real dates** (`datetime`), not ISO strings, so date operators, range queries and indexes work.

## Step 1 · Connect

In [1]:
import sys; sys.path.append("../src")
from config import *
import json, glob, inspect
import pandas as pd
from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import BulkWriteError, ServerSelectionTimeoutError
import mongo_transform as MT

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
try:
    client.admin.command("ping")
    print("Connected to MongoDB", client.server_info()["version"])
except ServerSelectionTimeoutError:
    raise SystemExit("MongoDB is not reachable. Start it with:  docker compose up -d mongo")

db = client[MONGO_DB]
pd.set_option("display.max_columns", 30, "display.width", 200)

Connected to MongoDB 7.0.45


## Step 2 · The transformation code
The functions that turn raw JSON into Mongo documents live in `src/mongo_transform.py` (unit-tested with `pytest`). Read them once – they *are* the schema design:

In [2]:
for fn in (MT.transform_product, MT.transform_transaction, MT.transform_session):
    print(inspect.getsource(fn))

def transform_product(p, lookup):
    cat = lookup.get(p["category_id"], {})
    return {
        "_id": p["product_id"],
        "product_id": p["product_id"],
        "name": p["name"],
        "category": {"category_id": p["category_id"], "name": cat.get("name"),
                     "avg_margin": cat.get("avg_margin")},
        "base_price": p["base_price"],
        "current_stock": p["current_stock"],
        "is_active": p["is_active"],
        "price_history": [{"price": h["price"], "date": parse_dt(h["date"])}
                          for h in p.get("price_history", [])],
        "creation_date": parse_dt(p["creation_date"]),
    }

def transform_transaction(t, product_category):
    items = []
    for it in t["items"]:
        items.append({**it, "category_id": product_category.get(it["product_id"])})
    return {
        "_id": t["transaction_id"],
        "transaction_id": t["transaction_id"],
        "session_id": t["session_id"],
        "user_id": t["user_id"],
        "

## Step 3 · Load the data
Loading order matters: **categories → products → users → transactions → sessions** (transactions need `product → category` to denormalise; users start with zeroed summaries).

The loader is **idempotent** (drops and recreates collections) and inserts in **batches of 5,000** with `ordered=False`, so one bad document cannot stop the load.

In [3]:
BATCH = 5000

def bulk_insert(collection, docs_iter, batch=BATCH):
    """Insert an iterable of documents in batches; returns number inserted."""
    buf, total = [], 0
    def flush():
        nonlocal buf, total
        if not buf: return
        try:
            total += len(collection.insert_many(buf, ordered=False).inserted_ids)
        except BulkWriteError as e:
            total += e.details["nInserted"]
            print("  ! write errors:", len(e.details["writeErrors"]), e.details["writeErrors"][0]["errmsg"][:120])
        buf = []
    for d in docs_iter:
        buf.append(d)
        if len(buf) >= batch: flush()
    flush()
    return total

# --- read raw files -------------------------------------------------------------------------
categories = json.load(open(RAW / "categories.json"))
products   = json.load(open(RAW / "products.json"))
users      = json.load(open(RAW / "users.json"))
txns       = json.load(open(RAW / "transactions.json"))
session_files = sorted(glob.glob(str(RAW / "sessions_*.json")))
print(f"Raw: {len(categories)} categories, {len(products)} products, {len(users)} users, {len(txns)} transactions, {len(session_files)} session files")

Raw: 25 categories, 5000 products, 10000 users, 50000 transactions, 2 session files


In [4]:
SESSION_FILES_TO_LOAD = len(session_files)   # all of them; lower this number for a faster/lighter load

for name in ["categories", "products", "users", "transactions", "sessions"]:
    db[name].drop()

cat_lookup = MT.build_category_lookup(categories)
product_category = {p["product_id"]: p["category_id"] for p in products}

with timer("categories"): n = bulk_insert(db.categories, (MT.transform_category(c, cat_lookup) for c in categories)); print("  categories  :", n)
with timer("products"):   n = bulk_insert(db.products,   (MT.transform_product(p, cat_lookup) for p in products));     print("  products    :", n)
with timer("users"):      n = bulk_insert(db.users,      (MT.transform_user(u) for u in users));                       print("  users       :", n)
with timer("transactions"): n = bulk_insert(db.transactions, (MT.transform_transaction(t, product_category) for t in txns)); print("  transactions:", n)

total = 0
with timer("sessions"):
    for f in session_files[:SESSION_FILES_TO_LOAD]:
        part = json.load(open(f))                       # one file at a time keeps memory low
        total += bulk_insert(db.sessions, (MT.transform_session(s) for s in part))
        print("  loaded", Path(f).name, "->", f"{total:,} sessions so far"); del part

  categories  : 25
[timer] categories: 0.16s
  products    : 5000
[timer] products: 0.34s
  users       : 10000
[timer] users: 0.45s
  transactions: 50000
[timer] transactions: 1.77s
  loaded sessions_0.json -> 100,000 sessions so far
  loaded sessions_1.json -> 200,000 sessions so far
[timer] sessions: 23.68s


In [5]:
# Sanity check: document counts must match the raw files
print({c: db[c].count_documents({}) for c in ["categories", "products", "users", "transactions", "sessions"]})
print("\nOne transaction document:"); print(json.dumps(db.transactions.find_one({}, {"_id": 0}), indent=2, default=str)[:900])

{'categories': 25, 'products': 5000, 'users': 10000, 'transactions': 50000, 'sessions': 200000}

One transaction document:
{
  "transaction_id": "txn_2fd090430c28",
  "session_id": "sess_50746d9dfe",
  "user_id": "user_007129",
  "timestamp": "2026-10-03 04:52:57",
  "items": [
    {
      "product_id": "prod_01609",
      "quantity": 2,
      "unit_price": 319.01,
      "subtotal": 638.02,
      "category_id": "cat_001"
    },
    {
      "product_id": "prod_04265",
      "quantity": 2,
      "unit_price": 243.6,
      "subtotal": 487.2,
      "category_id": "cat_018"
    }
  ],
  "n_items": 4,
  "subtotal": 1125.22,
  "discount": 168.78,
  "total": 956.44,
  "payment_method": "crypto",
  "status": "completed"
}


## Step 4 · Indexes
Indexes are chosen from the **query patterns** in Step 6 – not "index everything" (every index costs write speed and RAM).

| Collection | Index | Serves |
|---|---|---|
| transactions | `timestamp` | revenue over a period (range scans) |
| transactions | `user_id, timestamp` | a customer's order history |
| transactions | `items.product_id` *(multikey)* | "orders containing product X" |
| transactions | `items.category_id` *(multikey)* | category analytics |
| sessions | `user_id, start_time` | a user's journeys, newest first |
| sessions | `viewed_products` *(multikey)* | sessions that viewed product X |
| sessions | `conversion_status` | funnel counts |
| products | `category.category_id` | catalogue browsing by category |
| users | `purchase_summary.orders` | segmentation by frequency |

In [6]:
db.transactions.create_index([("timestamp", ASCENDING)])
db.transactions.create_index([("user_id", ASCENDING), ("timestamp", DESCENDING)])
db.transactions.create_index([("items.product_id", ASCENDING)])
db.transactions.create_index([("items.category_id", ASCENDING)])
db.sessions.create_index([("user_id", ASCENDING), ("start_time", DESCENDING)])
db.sessions.create_index([("viewed_products", ASCENDING)])
db.sessions.create_index([("conversion_status", ASCENDING)])
db.products.create_index([("category.category_id", ASCENDING)])
db.users.create_index([("purchase_summary.orders", ASCENDING)])
for c in ["transactions", "sessions", "products", "users"]:
    print(c, "->", [i["name"] for i in db[c].list_indexes()])

transactions -> ['_id_', 'timestamp_1', 'user_id_1_timestamp_-1', 'items.product_id_1', 'items.category_id_1']
sessions -> ['_id_', 'user_id_1_start_time_-1', 'viewed_products_1', 'conversion_status_1']
products -> ['_id_', 'category.category_id_1']
users -> ['_id_', 'purchase_summary.orders_1']


## Step 5 · Enrich user profiles with server-side pipelines (`$group` + `$merge`)
The assignment suggests *"embedding … summarized purchase history or preferences"* in user profiles. We compute those summaries **inside MongoDB** (no data leaves the server) and `$merge` them into `users`:

* **purchase_summary** – orders, total spent, average order value, first/last purchase
* **preferences.top_categories** – the 3 categories where the user spent the most (needs `$unwind` of items)

In [7]:
purchase_summary_pipeline = [
    {"$group": {"_id": "$user_id",
                "orders": {"$sum": 1},
                "total_spent": {"$sum": "$total"},
                "first_purchase": {"$min": "$timestamp"},
                "last_purchase": {"$max": "$timestamp"}}},
    {"$project": {"_id": 1,
                  "purchase_summary": {"orders": "$orders",
                                       "total_spent": {"$round": ["$total_spent", 2]},
                                       "avg_order_value": {"$round": [{"$divide": ["$total_spent", "$orders"]}, 2]},
                                       "first_purchase": "$first_purchase",
                                       "last_purchase": "$last_purchase"}}},
    {"$merge": {"into": "users", "on": "_id", "whenMatched": "merge", "whenNotMatched": "discard"}},
]

top_categories_pipeline = [
    {"$unwind": "$items"},
    {"$group": {"_id": {"u": "$user_id", "c": "$items.category_id"}, "spend": {"$sum": "$items.subtotal"}}},
    {"$sort": {"spend": -1}},
    {"$group": {"_id": "$_id.u", "cats": {"$push": {"category_id": "$_id.c", "spend": {"$round": ["$spend", 2]}}}}},
    {"$project": {"preferences": {"top_categories": {"$slice": ["$cats", 3]}}}},
    {"$merge": {"into": "users", "on": "_id", "whenMatched": "merge", "whenNotMatched": "discard"}},
]

with timer("purchase_summary"):   list(db.transactions.aggregate(purchase_summary_pipeline, allowDiskUse=True))
with timer("top_categories"):     list(db.transactions.aggregate(top_categories_pipeline, allowDiskUse=True))
print(json.dumps(db.users.find_one({"purchase_summary.orders": {"$gt": 2}}, {"_id": 0}), indent=2, default=str))

[timer] purchase_summary: 0.81s
[timer] top_categories: 1.12s
{
  "user_id": "user_000001",
  "geo_data": {
    "city": "Michaelstad",
    "state": "MO",
    "country": "DJ"
  },
  "registration_date": "2026-06-12 20:04:39",
  "last_active": "2026-07-22 20:03:37",
  "purchase_summary": {
    "orders": 3,
    "total_spent": 5236.33,
    "avg_order_value": 1745.44,
    "first_purchase": "2026-08-14 08:14:07",
    "last_purchase": "2026-10-23 07:11:03"
  },
  "preferences": {
    "top_categories": [
      {
        "category_id": "cat_001",
        "spend": 1388.06
      },
      {
        "category_id": "cat_004",
        "spend": 1086.09
      },
      {
        "category_id": "cat_013",
        "spend": 966.74
      }
    ]
  }
}


## Step 6 · Analytical aggregation pipelines
The assignment requires **two** non-trivial pipelines; we implement **five** so the report and charts have solid material.
Each pipeline is shown, explained, run, timed, and exported to `outputs/tables/`.

Helper to run + time + save:

In [8]:
def run_pipeline(collection, pipeline, name=None, show=10):
    with timer(name or "pipeline"):
        rows = list(collection.aggregate(pipeline, allowDiskUse=True))
    df = pd.DataFrame(rows)
    if name:
        df.to_csv(TAB / f"{name}.csv", index=False)
    display(df.head(show)); return df

### Pipeline 1 – Top-selling products (units, revenue, distinct buyers) – *Product popularity*
`$unwind` the embedded items → `$group` per product → `$sort/$limit` → `$lookup` only the **10 winners** into `products` to get names/categories (cheap because lookup happens *after* the limit).

In [9]:
p1 = [
    {"$unwind": "$items"},
    {"$group": {"_id": "$items.product_id",
                "units_sold": {"$sum": "$items.quantity"},
                "revenue": {"$sum": "$items.subtotal"},
                "buyers": {"$addToSet": "$user_id"}}},
    {"$addFields": {"n_buyers": {"$size": "$buyers"}}},
    {"$sort": {"units_sold": -1, "revenue": -1}},
    {"$limit": 20},
    {"$lookup": {"from": "products", "localField": "_id", "foreignField": "_id", "as": "p"}},
    {"$unwind": "$p"},
    {"$project": {"_id": 0, "product_id": "$_id", "name": "$p.name", "category": "$p.category.name",
                  "units_sold": 1, "revenue": {"$round": ["$revenue", 2]}, "n_buyers": 1}},
]
top_products = run_pipeline(db.transactions, p1, "mongo_top_products")

[timer] mongo_top_products: 0.26s


,units_sold,n_buyers,product_id,name,category,revenue
0,102,46,prod_01751,Diverse Motivating Time-Frame,Baldwin Ltd,33779.34
1,87,37,prod_04285,Polarized Hybrid Projection,"Ellis, Baker and Wright",1786.11
2,85,38,prod_04853,Mandatory Discrete Internet Solution,Hancock Inc,20078.70
3,83,41,prod_03450,Assimilated Full-Range Algorithm,Preston Ltd,31225.43
4,83,40,prod_02747,Proactive Modular Analyzer,Kelly-Santiago,5386.70
5,82,36,prod_03080,Public-Key Dedicated Complexity,"Moore, Becker and Carlson",12891.22
6,81,34,prod_01889,Triple-Buffered Value-Added Workforce,Powell PLC,37582.38
7,81,35,prod_03640,Virtual Bottom-Line Infrastructure,Cameron-Parsons,30624.48
8,80,40,prod_00823,Enterprise-Wide Local Analyzer,Hancock Inc,34399.20
9,80,37,prod_01551,Automated 4Thgeneration Process Improvement,Spence PLC,5902.40


### Pipeline 2 – Most-viewed products & view→purchase ratio – *Product popularity (browsing side)*
Uses the **sessions** collection (`viewed_products` is already a de-duplicated array per session, so `$unwind` counts *sessions that viewed* each product). A second `$lookup` is **not** needed: we join the sales figures in Python/Spark later.

In [10]:
p2 = [
    {"$unwind": "$viewed_products"},
    {"$group": {"_id": "$viewed_products", "sessions_viewed": {"$sum": 1}, "viewers": {"$addToSet": "$user_id"}}},
    {"$project": {"_id": 0, "product_id": "$_id", "sessions_viewed": 1, "unique_viewers": {"$size": "$viewers"}}},
    {"$sort": {"sessions_viewed": -1}},
    {"$limit": 5000},
]
most_viewed = run_pipeline(db.sessions, p2, "mongo_most_viewed")

[timer] mongo_most_viewed: 1.10s


,sessions_viewed,product_id,unique_viewers
0,193,prod_04857,191
1,192,prod_02991,191
2,190,prod_02412,186
3,186,prod_01481,185
4,186,prod_01973,185
5,185,prod_00199,180
6,185,prod_03323,185
7,184,prod_02862,183
8,184,prod_00073,184
9,184,prod_03012,183


### Pipeline 3 – Net revenue and estimated gross profit by category and month – *Revenue analytics*
Non-trivial because **discounts are stored per order, not per item**. We allocate each order's discount to its items **proportionally** (`factor = total / subtotal`), then group by `(category, month)`. Margin comes from the category's average subcategory margin (see Limitations: products have no `subcategory_id`).

In [11]:
p3 = [
    {"$addFields": {"factor": {"$cond": [{"$gt": ["$subtotal", 0]}, {"$divide": ["$total", "$subtotal"]}, 1]}}},
    {"$unwind": "$items"},
    {"$group": {"_id": {"category_id": "$items.category_id",
                        "month": {"$dateToString": {"format": "%Y-%m", "date": "$timestamp"}}},
                "gross_revenue": {"$sum": "$items.subtotal"},
                "net_revenue": {"$sum": {"$multiply": ["$items.subtotal", "$factor"]}},
                "units": {"$sum": "$items.quantity"}}},
    {"$lookup": {"from": "categories", "localField": "_id.category_id", "foreignField": "_id", "as": "cat"}},
    {"$unwind": "$cat"},
    {"$project": {"_id": 0, "category_id": "$_id.category_id", "category": "$cat.name", "month": "$_id.month",
                  "units": 1, "gross_revenue": {"$round": ["$gross_revenue", 2]},
                  "net_revenue": {"$round": ["$net_revenue", 2]},
                  "est_gross_profit": {"$round": [{"$multiply": ["$net_revenue", "$cat.avg_profit_margin"]}, 2]}}},
    {"$sort": {"month": 1, "net_revenue": -1}},
]
rev_cat_month = run_pipeline(db.transactions, p3, "mongo_revenue_by_category_month")
print("Top categories overall:")
display(rev_cat_month.groupby("category")[["net_revenue", "est_gross_profit"]].sum().sort_values("net_revenue", ascending=False).head(5).round(0))

[timer] mongo_revenue_by_category_month: 0.28s


,units,category_id,category,month,gross_revenue,net_revenue,est_gross_profit
0,2311,cat_004,Kelly-Santiago,2026-08,585774.51,571522.88,117733.71
1,2109,cat_005,Cameron-Parsons,2026-08,554362.13,538586.69,148111.34
2,2161,cat_014,"Anderson, Walls and Duncan",2026-08,543673.38,530147.95,129886.25
3,2201,cat_018,Phillips Inc,2026-08,536201.57,523553.36,129841.23
4,2287,cat_017,Hancock Inc,2026-08,537163.28,523413.01,151789.77
5,2031,cat_024,"Patterson, Smith and Jones",2026-08,535006.68,521887.51,140909.63
6,2053,cat_006,"Ellis, Baker and Wright",2026-08,534470.29,521229.34,111543.08
7,2068,cat_021,Mitchell-Kim,2026-08,533254.71,519665.55,113027.26
8,2120,cat_012,Carney-Santos,2026-08,530014.47,518111.46,134708.98
9,1994,cat_015,Owens-Daniel,2026-08,529701.90,517114.46,136518.22


Top categories overall:


,net_revenue,est_gross_profit
category,,
Kelly-Santiago,2597128.0,535008.0
Cameron-Parsons,2427289.0,667504.0
"Anderson, Walls and Duncan",2425305.0,594200.0
"Patterson, Smith and Jones",2316601.0,625482.0
"Ellis, Baker and Wright",2294903.0,491109.0


### Pipeline 4 – Customer segmentation by purchase frequency – *Basic user segmentation*
Reads the **embedded `purchase_summary`** created in Step 5 – a good example of why embedding pays off: a single-collection `$bucket` over 10 k users instead of a join over 50 k+ transactions.
(`age group` – suggested in the assignment – is impossible: the data has no age. Frequency is the natural replacement.)

In [12]:
p4 = [
    {"$bucket": {"groupBy": "$purchase_summary.orders",
                 "boundaries": [0, 1, 2, 4, 7, 11, 1000],
                 "default": "other",
                 "output": {"users": {"$sum": 1},
                            "avg_spend": {"$avg": "$purchase_summary.total_spent"},
                            "total_spend": {"$sum": "$purchase_summary.total_spent"},
                            "avg_order_value": {"$avg": "$purchase_summary.avg_order_value"}}}},
    {"$project": {"_id": 0,
                  "orders_bucket": {"$switch": {"branches": [
                      {"case": {"$eq": ["$_id", 0]},  "then": "0 orders (never bought)"},
                      {"case": {"$eq": ["$_id", 1]},  "then": "1 order"},
                      {"case": {"$eq": ["$_id", 2]},  "then": "2-3 orders"},
                      {"case": {"$eq": ["$_id", 4]},  "then": "4-6 orders"},
                      {"case": {"$eq": ["$_id", 7]},  "then": "7-10 orders"},
                      {"case": {"$eq": ["$_id", 11]}, "then": "11+ orders"}], "default": "other"}},
                  "users": 1, "avg_spend": {"$round": ["$avg_spend", 2]},
                  "total_spend": {"$round": ["$total_spend", 2]},
                  "avg_order_value": {"$round": ["$avg_order_value", 2]}}},
]
freq_segments = run_pipeline(db.users, p4, "mongo_user_frequency_segments")

[timer] mongo_user_frequency_segments: 0.04s


,users,orders_bucket,avg_spend,total_spend,avg_order_value
0,58,0 orders (never bought),0.00,0.00,0.00
1,352,1 order,1074.34,378169.42,1074.34
2,2224,2-3 orders,2807.28,6243398.78,1066.41
3,4992,4-6 orders,5325.38,26584286.59,1079.43
4,2235,7-10 orders,8476.11,18944096.96,1081.50
5,139,11+ orders,12784.34,1777023.91,1098.47


### Pipeline 5 – Daily revenue + orders (time series) and Session conversion by device & referrer
Two short pipelines that feed the charts. (`conversion_status` is used here as-is; notebook 05 reconciles it with real transactions, because we found in notebook 00 that ~38 % of "converted" sessions have no order.)

In [13]:
p5a = [
    {"$group": {"_id": {"$dateToString": {"format": "%Y-%m-%d", "date": "$timestamp"}},
                "orders": {"$sum": 1}, "revenue": {"$sum": "$total"}, "avg_order_value": {"$avg": "$total"}}},
    {"$project": {"_id": 0, "date": "$_id", "orders": 1, "revenue": {"$round": ["$revenue", 2]},
                  "avg_order_value": {"$round": ["$avg_order_value", 2]}}},
    {"$sort": {"date": 1}},
]
daily = run_pipeline(db.transactions, p5a, "mongo_daily_revenue", show=5)

p5b = [
    {"$group": {"_id": {"device": "$device_profile.type", "referrer": "$referrer"},
                "sessions": {"$sum": 1},
                "converted": {"$sum": {"$cond": [{"$eq": ["$conversion_status", "converted"]}, 1, 0]}},
                "avg_duration": {"$avg": "$duration_seconds"}}},
    {"$project": {"_id": 0, "device": "$_id.device", "referrer": "$_id.referrer", "sessions": 1, "converted": 1,
                  "conv_rate_pct": {"$round": [{"$multiply": [{"$divide": ["$converted", "$sessions"]}, 100]}, 2]},
                  "avg_duration": {"$round": ["$avg_duration", 0]}}},
    {"$sort": {"conv_rate_pct": -1}},
]
device_ref = run_pipeline(db.sessions, p5b, "mongo_conversion_by_device_referrer")

[timer] mongo_daily_revenue: 0.12s


,orders,date,revenue,avg_order_value
0,285,2026-08-11,308844.68,1083.67
1,567,2026-08-12,592528.49,1045.02
2,491,2026-08-13,515567.33,1050.04
3,557,2026-08-14,621752.23,1116.25
4,543,2026-08-15,600055.64,1105.07


[timer] mongo_conversion_by_device_referrer: 0.49s


,sessions,converted,device,referrer,conv_rate_pct,avg_duration
0,13154,2764,mobile,search_engine,21.01,1806.0
1,13436,2802,tablet,social,20.85,1812.0
2,13253,2757,desktop,affiliate,20.80,1801.0
3,13262,2745,tablet,search_engine,20.70,1838.0
4,13209,2721,tablet,direct,20.60,1810.0
5,13581,2787,desktop,direct,20.52,1814.0
6,13446,2750,mobile,direct,20.45,1810.0
7,13230,2683,desktop,search_engine,20.28,1806.0
8,13317,2697,mobile,affiliate,20.25,1814.0
9,13327,2692,tablet,affiliate,20.20,1823.0


## Step 7 · Prove the indexes help (`explain`)
`explain("executionStats")` shows whether MongoDB scanned the whole collection (`COLLSCAN`) or used an index (`IXSCAN`). We compare the same query with and without its index – numbers for the report.

In [14]:
def explain_find(coll, flt, hint=None):
    cur = coll.find(flt)
    if hint: cur = cur.hint(hint)
    ex = cur.explain()["executionStats"]
    return {"nReturned": ex["nReturned"], "docsExamined": ex["totalDocsExamined"], "ms": ex["executionTimeMillis"]}

some_user = users[0]["user_id"]
query = {"user_id": some_user}
with_idx   = explain_find(db.transactions, query, hint=[("user_id", ASCENDING), ("timestamp", DESCENDING)])
without    = explain_find(db.transactions, query, hint=[("$natural", 1)])      # force a collection scan
display(pd.DataFrame({"with index (IXSCAN)": with_idx, "forced COLLSCAN": without}).T)

,nReturned,docsExamined,ms
with index (IXSCAN),4,4,3
forced COLLSCAN,4,50000,27


## Step 8 · Two more document-model queries (journey analysis)
The assignment's *Example Query Opportunity* for MongoDB: **analyse a complete user journey in a single document** – referrer, device and all page views, with no joins.

In [15]:
# Most common first-three-page paths of CONVERTED sessions (a true "journey" question)
journey = [
    {"$match": {"conversion_status": "converted"}},
    {"$project": {"path": {"$map": {"input": {"$slice": ["$page_views", 4]}, "as": "p", "in": "$$p.page_type"}}}},
    {"$group": {"_id": "$path", "sessions": {"$sum": 1}}},
    {"$sort": {"sessions": -1}}, {"$limit": 8},
    {"$project": {"_id": 0, "first_pages": {"$reduce": {"input": "$_id", "initialValue": "",
                  "in": {"$concat": ["$$value", {"$cond": [{"$eq": ["$$value", ""]}, "", " > "]}, "$$this"]}}}, "sessions": 1}},
]
display(pd.DataFrame(db.sessions.aggregate(journey, allowDiskUse=True)))

# A single user's latest session, fully reconstructed from ONE document
display(pd.json_normalize(db.sessions.find_one({"user_id": some_user}, sort=[("start_time", -1)])["page_views"]))

,sessions,first_pages
0,3261,category_listing > product_detail > cart > che...
1,2850,search > product_detail > cart > checkout
2,1837,home > category_listing > product_detail > cart
3,1431,category_listing > product_detail > product_de...
4,1342,home > category_listing > product_detail > pro...
5,1322,category_listing > product_detail > category_l...
6,1218,search > product_detail > category_listing > p...
7,1164,search > product_detail > product_detail > cart


,timestamp,page_type,product_id,category_id,view_duration
0,2026-10-30 03:08:45,home,NaN,NaN,152
1,2026-10-30 03:11:17,search,NaN,NaN,97
2,2026-10-30 03:12:54,product_detail,prod_01184,cat_020,300
3,2026-10-30 03:17:54,category_listing,NaN,cat_003,37
4,2026-10-30 03:18:31,product_detail,prod_00117,cat_005,581
5,2026-10-30 03:28:12,home,NaN,NaN,226
6,2026-10-30 03:31:58,category_listing,NaN,cat_008,455
7,2026-10-30 03:39:33,product_detail,prod_00369,cat_006,144
8,2026-10-30 03:41:57,product_detail,prod_00313,cat_016,803


### ✅ What you should have after this notebook
* 5 collections in database `ecommerce` + 9 indexes.
* User documents enriched with `purchase_summary` and `preferences`.
* CSV results in `outputs/tables/`: `mongo_top_products`, `mongo_most_viewed`, `mongo_revenue_by_category_month`, `mongo_user_frequency_segments`, `mongo_daily_revenue`, `mongo_conversion_by_device_referrer`.
* Timing numbers (printed) and the index-vs-collscan table → paste into the report.

📝 **For the report (Design Document):** copy the five modelling rules from the top of this notebook and, for every collection, state *embedded vs referenced* + *why* + *which query it serves*.

➡ Next: `02_hbase_modeling_and_queries.ipynb`.